# ASL Index & Climatology Workflow

This notebook drives the ASL (Amundsen Sea Low) analysis workflow, which includes:
1. Identifying ASL locations and generating indices from PSL data.
2. Generating seasonal and annual cycle time series (climatology step 1).
3. Calculating mean climatology and bias against observation/reanalysis (climatology step 2).
4. Performing ASL lead-lag regression and correlation analysis.

In [ ]:
import os
import sys
import glob
import collections

# Ensure our shared libraries are discoverable
sys.path.append("..")
from polar_utils.common import Case
from polar_utils.asl import (
    run_asl_index_generation,
    run_asl_leadlag_analysis,
    run_asl_index_clim_step1,
    run_asl_index_clim_step2
)

In [ ]:
# Global Configuration Paths
top_path = "/lcrc/group/e3sm2/ac.szhang/E3SMv21_testings/v3_polar_paper"
run_path = "/lcrc/group/e3sm/ac.szhang/acme_scratch/data/pcmdi/model/monthly"
run_mask = "/lcrc/group/e3sm/ac.szhang/acme_scratch/data/pcmdi/model/fixed/sftlf"
model_root = "/lcrc/group/e3sm2/ac.szhang/E3SMv21_testings"

asl_region = {'west': 170., 'east': 298., 'south': -80., 'north': -60.}
asl_min_dist = 5
asl_num_peak = 3
asl_exc_bord = False
l_check_asl_region = False
l_allow_no_asl = False

## 1. Run ASL Index Generation

In [ ]:
# Define out and figure directories
out_path = os.path.join(top_path, "fig_data", "asl_analysis", "raw_index")
fig_path = os.path.join(top_path, "polar_analysis", "3_asl_analysis", "asl_index_ts", "figure")

# 1.1 Historical E3SM run (using the combined SORRM simulation)
case_dict = collections.OrderedDict()
sorrm_files = os.path.join(model_root, "v2_1.SORRM.histssp370_0701/post/atm/180x360_aave/ts/monthly/10yr/PSL_*.nc")
case_dict["v2_1-SORRM"] = Case(sorrm_files, "PSL", "blue", "v2_1-SORRM")
case_dict['mask'] = Case(os.path.join(run_mask, "Amon/e3sm.historical.v2_1-SORRM.fx.sftlf.nc"), "sftlf", "blue", "v2_1-SORRM")
run_asl_index_generation(fig_path, out_path, "e3sm", "historical", "0701", "asl_scotthoskingv3", "195101-201412", 
                         case_dict, asl_region, 1, 12, asl_exc_bord, l_check_asl_region, l_allow_no_asl)

# 1.2 Control E3SM run
case_dict = collections.OrderedDict()
control_files = os.path.join(model_root, "v2_1.SORRM.control/post/atm/180x360_aave/ts/monthly/50yr/PSL_*.nc")
case_dict["v2_1-SORRM"] = Case(control_files, "PSL", "blue", "v2_1-SORRM")
case_dict['mask'] = Case(os.path.join(run_mask, "Amon/e3sm.piControl.v2_1-SORRM.fx.sftlf.nc"), "sftlf", "blue", "v2_1-SORRM")
run_asl_index_generation(fig_path, out_path, "e3sm", "piControl", "1950", "asl_scotthoskingv3", "080101-100012", 
                         case_dict, asl_region, 1, 12, asl_exc_bord, l_check_asl_region, l_allow_no_asl)

# 1.3 Future E3SM SSP370 run (using the same combined SORRM simulation)
case_dict = collections.OrderedDict()
case_dict["v2_1-SORRM"] = Case(sorrm_files, "PSL", "blue", "v2_1-SORRM")
case_dict['mask'] = Case(os.path.join(run_mask, "Amon/e3sm.ssp370.v2_1-SORRM.fx.sftlf.nc"), "sftlf", "blue", "v2_1-SORRM")
run_asl_index_generation(fig_path, out_path, "e3sm", "ssp370", "0701", "asl_scotthoskingv3", "201501-210012", 
                         case_dict, asl_region, 1, 12, asl_exc_bord, l_check_asl_region, l_allow_no_asl)

# 1.4 Observation/Analysis Runs (NOAA_20C and ERA5)
for obs_name in ["NOAA_20C", "ERA5"]:
    case_dict = collections.OrderedDict()
    obs_file = glob.glob(os.path.join(run_path, "analysis", obs_name, "Amon/psl/analysis.historical.{}.en00.*.nc".format(obs_name)))[0]
    case_dict[obs_name] = Case(obs_file, "PSL" if obs_name == "NOAA_20C" else "psl", "blue", obs_name)
    mask_file = os.path.join(run_mask, "Amon/analysis.historical.{}.fx.sftlf.nc".format(obs_name))
    case_dict['mask'] = Case(mask_file, "sftlf", "blue", obs_name)
    run_asl_index_generation(fig_path, out_path, "analysis", "historical", "en00", "asl_scotthoskingv3", "195001-201412", 
                             case_dict, asl_region, 1, 12, asl_exc_bord, l_check_asl_region, l_allow_no_asl)

## 2. Run ASL Climatology Index Calculations (Step 1 & Step 2)

In [ ]:
# Directories for Climatology
raw_index_path = os.path.join(top_path, "fig_data", "asl_analysis", "raw_index")
ts_index_path = os.path.join(top_path, "fig_data", "asl_analysis", "ts_index")
clim_index_path = os.path.join(top_path, "fig_data", "asl_analysis", "clim_index")

# Step 1: Calculate Seasonal/Annual Cycle
mips = ["analysis", "cmip6", "e3sm"]
exps = ["historical", "ssp370", "piControl"]
ver = "asl_scotthoskingv3"
seasons = ['ANN', 'DJF', 'JJA', 'MAM', 'SON', 'AC', 'Monthly']
indices = ['lon', 'lat', 'ActCenPres', 'SectorPres', 'RelCenPres']

run_asl_index_clim_step1(mips, exps, ver, seasons, indices, raw_index_path, ts_index_path)

# Step 2: Compare against Observation Climatologies
obs_sets = ['NOAA_20C.en00', 'ERA5.en00']
obs_mips = ['analysis', 'analysis']
periods = ["1950-2014", "1979-2014"]
test_mips = ["cmip6", "e3sm"]
test_exps = ["historical"]
clim_seasons = ['ANN', 'DJF', 'JJA', 'MAM', 'SON', 'AC']

run_asl_index_clim_step2(obs_sets, obs_mips, test_mips, test_exps, periods, clim_seasons, indices, ts_index_path, clim_index_path)

## 3. Run ASL Lead-Lag Analysis

In [ ]:
# Directories for Lead-Lag
lead_lag_out = os.path.join(top_path, "fig_data", "asl_analysis", "lead_lag")
lead_lag_fig = os.path.join(top_path, "polar_analysis", "3_asl_analysis", "lead_lag_analysis", "figure")

# E3SM Historical
case_dict = collections.OrderedDict()
case_dict["v2_1-SORRM"] = Case(sorrm_files, "PSL", "blue", "v2_1-SORRM")
run_asl_leadlag_analysis(lead_lag_fig, lead_lag_out, "e3sm", "historical", "0701", "asl_scotthoskingv3", "195101-201412", 
                         case_dict, asl_region, "RelCenPres", l_check_asl_region)